# Meridional overturning circulation

The overturning streamfunction in depth space and in density
space.

In depth space CMIP6 publishes the streamfunction directly as
`msftyz`, so there is little to do.

In density space there is a real methodological gap. MOM bins the
meridional transport into density layers *online*, into
`ty_trans_rho`. CMIP6 has no such variable, so the only option is
to rebin the z-coordinate `vmo` offline using monthly `thetao`
and `so`. Rebinning monthly means cannot recover the within-month
correlation between transport and density, so **the eddy
contribution is underestimated**. Do not present this as
equivalent to `ty_trans_rho`.

Latitudes north of 65N are dropped: on a tripolar grid the local
y-direction there is not northward, so a zonal sum of `vmo` is
meaningless
([cosima-recipes#510](https://github.com/COSIMA/cosima-recipes/issues/510)).

---

Translated from the COSIMA recipe
[`Overturning_Circulation.ipynb`](https://github.com/COSIMA/cosima-recipes/blob/main/03-Advanced-Recipes/Overturning_Circulation.ipynb)
onto CMORised CMIP6 data, using ESMValCore's `Dataset` and
preprocessors in place of the ACCESS-NRI intake catalog.

This notebook is standalone: it needs only `esmvalcore`,
`iris`, `numpy`, `matplotlib`, `cartopy` and `gsw`. The
equivalent ESMValTool recipe is
`recipes/recipe_cosima_*.yml`.

In [ ]:
import numpy as np
import iris
import iris.analysis.cartography
import iris.plot as iplt
import matplotlib.pyplot as plt

from esmvalcore.dataset import Dataset
import gsw
from esmvalcore.preprocessor import climate_statistics

## Start a dask cluster

Every COSIMA recipe opens with one, and these need one for the same
reason, though it is worth being precise about *which* part it speeds up.

**What the cluster does accelerate.** Everything from `Dataset.load()`
through the ESMValCore preprocessors — reading multi-decade files,
`regrid`, `climate_statistics`, `extract_region` — is lazy and chunked.
That is the expensive part of these notebooks and it is what the workers
parallelise. Without a client, dask falls back to its threaded scheduler;
ESMValCore itself warns that the distributed scheduler is preferable.

**What it does not.** Once the preprocessed cube is reduced, the
diagnostic arithmetic realises it into a plain masked array and proceeds
eagerly in numpy. That is deliberate: contour tracing, per-column
remapping and density binning are iterative and index-based, so they do
not express well as array graphs and would be slower, not faster, spread
over workers. Realising *after* the time-mean is what keeps that
affordable — realising a 20-year daily 3-D field before reducing it would
not be.

So: size the cluster for the load-and-preprocess stage. One thread per
worker, as the COSIMA recipes use — with chunks this large, per-worker
memory binds well before thread count does.

On ARE, `Client(threads_per_worker=1)` picks up the cores you requested
for the session. Pass `n_workers=` and `memory_limit=` to be explicit.

**Running the equivalent recipe instead?** Do not start a client there.
ESMValTool manages dask itself, and it defaults to the threaded
scheduler; switch it on in your ESMValTool configuration:

```yaml
dask:
  use: local_distributed
```

In [ ]:
import logging
import os

from dask.distributed import Client

# distributed logs every worker registration and teardown at INFO,
# which is hundreds of lines per notebook and buries everything the
# diagnostic prints. Warnings and errors still come through.
for _name in ("distributed", "distributed.scheduler", "distributed.nanny",
              "distributed.worker", "distributed.core",
              "distributed.batched", "distributed.utils_perf"):
    logging.getLogger(_name).setLevel(logging.WARNING)

# Client.current() reuses a cluster if this cell is re-run, instead of
# quietly starting a second one alongside the first.
try:
    worker_count = max(1, int(os.environ.get("PBS_NCPUS", "1")))
except ValueError:
    worker_count = 1
    print("PBS_NCPUS is not an integer; using one worker")
try:
    client = Client.current()
except ValueError:
    try:
        # Separate worker processes: real memory isolation, and what
        # you want on ARE or any batch node.
        client = Client(n_workers=worker_count, threads_per_worker=1)
    except (RuntimeError, OSError) as error:
        # Some containers and locked-down environments cannot spawn
        # worker subprocesses. In-process workers still give a
        # distributed scheduler and a dashboard, just no isolation.
        print(f'process workers unavailable ({error}); '
              f'using in-process workers instead')
        client = Client(n_workers=worker_count, threads_per_worker=1,
                        processes=False)

active_workers = len(client.scheduler_info()['workers'])
print('dashboard:', client.dashboard_link)
print(f'workers: {active_workers} active (requested {worker_count})')
client

## Why `vmo` must not be regridded

The obvious way to translate this notebook is to `regrid`
everything onto a tidy 1x1 grid and carry on. That is wrong, and
wrong in a way that does not announce itself.

`vmo` is **extensive**: it is kg s$^{-1}$ through one cell face,
not a flux per unit area. Interpolation computes a plausible
*value* for each new cell but does not rescale it for the new
cell's size, so the sum changes with the number of cells:

| regrid | total transport |
|---|---|
| none | 8348 Sv |
| 1x1 → 0.5x0.5 | 31911 Sv (3.8x) |
| 1x1 → 0.25x0.25 | 127644 Sv (15.3x) |

Nothing errors, and an overturning cell of the wrong strength
still looks like an overturning cell.

Staying native is also the *better* choice here, not merely the
safe one:

- on a tripolar grid the rows follow latitude circles closely
  south of 65N, so summing along the `i` index already **is** a
  zonal integral;
- `vmo`, `thetao` and `so` share that grid, so binning the
  transport by the density of its own cell is exact rather than
  an artefact of two separate interpolations.

North of 65N neither holds — the local `y` direction stops being
northward — which is why the plot is cut at 65N.

If you do need transports on a regular grid, divide by the cell
face area first, regrid the resulting flux density, then multiply
by the new face area.

## Load the data

In [ ]:
common = dict(project='CMIP6', mip='Omon', exp='historical',
              ensemble='r1i1p1f1', grid='gn',
              timerange='1990/2009')

vmo = Dataset(short_name='vmo', dataset='ACCESS-CM2', **common)
thetao = vmo.copy(short_name='thetao')
so = vmo.copy(short_name='so')
msftyz = vmo.copy(short_name='msftyz')

def prepare(dataset):
    """Time-mean only -- no regrid: see above."""
    return climate_statistics(dataset.load(), operator='mean',
                              period='full')

# All three stay on the model's native grid, which they already
# share, so the cell-by-cell density binning is exact.
vmo_cube = prepare(vmo)
thetao_cube = prepare(thetao)
so_cube = prepare(so)

## Helpers

In [ ]:
# The helpers below are lifted verbatim from
# cosima_cmip/cosima_common.py by notebooks/build_notebooks.py, so
# this notebook runs exactly the same code as the ESMValTool
# diagnostic script. Edit them there, then rebuild.
import contextlib
import warnings

try:  # TEOS-10, needed by the density helpers
    import gsw
except ImportError:
    gsw = None


import logging

LOGGER = logging.getLogger("cosima_cmip")


EARTH_RADIUS = 6371000.0  # m, matches iris.analysis.cartography


RHO0 = 1035.0  # kg m-3, Boussinesq reference density used by MOM5/MOM6


FILL_VALUE_THRESHOLD = 1.0e19


def masked_data(cube, dtype=float):
    """Realised, fully masked data of a cube.

    Always go through this rather than ``cube.core_data()``.  The latter
    returns a *lazy* dask array, on which ``numpy.ma`` operations are
    silently no-ops: the mask is not applied, and fill values flow
    straight into the arithmetic.  This realises the data, applies any
    mask the file carries, and additionally masks non-finite values and
    unconverted fill values, which real CMOR files do contain.
    """
    data = np.ma.masked_invalid(np.ma.asarray(cube.data, dtype=dtype))
    return np.ma.masked_where(
        np.abs(np.ma.filled(data, 0.0)) >= FILL_VALUE_THRESHOLD, data)


def _coord_points(cube, name):
    """1D points for ``name``, taking column/row 0 of a 2D aux coord."""
    coord = cube.coord(name)
    if coord.ndim == 1:
        return coord.points
    points = coord.points
    # On a curvilinear grid latitude varies down a column and longitude
    # along a row; take the slice that actually varies.
    if name == "latitude":
        return points[:, 0]
    return points[0, :]


def lat_1d(cube):
    """Representative 1D latitude, also for 2D (curvilinear) coordinates."""
    return _coord_points(cube, "latitude")


def lon_1d(cube):
    """Representative 1D longitude, also for 2D coordinates."""
    return _coord_points(cube, "longitude")


def depth_coord_name(cube):
    """Name of the vertical coordinate, or None for a 2D field."""
    for name in ("depth", "ocean_sigma_z", "olevel", "lev"):
        if cube.coords(name):
            return name
    for coord in cube.coords(dim_coords=True):
        if coord.units.is_convertible("m") and coord.name() != "latitude":
            return coord.name()
    return None


def time_mean(cube):
    """Average over time if the cube still carries a time coordinate."""
    if cube.coords("time") and cube.coord("time").shape[0] > 1:
        return cube.collapsed("time", iris.analysis.MEAN)
    if cube.coords("time"):
        return iris.util.squeeze(cube)
    return cube


def bin_centres(bins):
    """Midpoints of a monotonic array of bin edges."""
    bins = np.asarray(bins, dtype=float)
    return 0.5 * (bins[1:] + bins[:-1])


def bin_sum(values, bin_variable, bins, axis=None):
    """Sum ``values`` into ``bins`` of ``bin_variable``.

    ``values`` and ``bin_variable`` must have the same shape.  Returns
    an array whose first axis runs over the bin centres and whose
    remaining axes are the axes of ``values`` *not* listed in ``axis``.
    With ``axis=None`` everything is collapsed into a 1D profile.

    This is ``xhistogram.xarray.histogram(bin_variable, bins=bins,
    weights=values, dim=axis)`` without the xarray dependency.
    """
    values = np.ma.masked_invalid(np.ma.asarray(values, dtype=float))
    bin_variable = np.ma.masked_invalid(
        np.ma.asarray(bin_variable, dtype=float))
    bins = np.asarray(bins, dtype=float)
    n_bins = bins.size - 1

    if axis is None:
        axis = tuple(range(values.ndim))
    elif isinstance(axis, int):
        axis = (axis,)
    axis = tuple(a % values.ndim for a in axis)
    keep = tuple(a for a in range(values.ndim) if a not in axis)

    moved = np.ma.transpose(values, keep + axis)
    moved_bin = np.ma.transpose(bin_variable, keep + axis)
    kept_shape = moved.shape[:len(keep)]
    flat = moved.reshape(int(np.prod(kept_shape)) if keep else 1, -1)
    flat_bin = moved_bin.reshape(flat.shape)

    out = np.zeros((n_bins,) + kept_shape)
    flat_out = out.reshape(n_bins, -1)
    # np.digitize returns 0 below the first edge and n_bins+1 above the
    # last, so a bin index of 1..n_bins is inside the range.
    for column in range(flat.shape[0]):
        data = flat[column]
        coordinate = flat_bin[column]
        valid = ~np.ma.getmaskarray(data) & ~np.ma.getmaskarray(coordinate)
        if not valid.any():
            continue
        index = np.digitize(
            np.ma.filled(coordinate, np.nan)[valid], bins)
        inside = (index >= 1) & (index <= n_bins)
        flat_out[:, column] = np.bincount(
            index[inside] - 1,
            weights=np.ma.filled(data, 0.0)[valid][inside],
            minlength=n_bins)
    return out


def rebin_to_density(transport, density, bins, depth_axis=0):
    """Rebin a z-coordinate transport onto density bins.

    ``Transformation_from_Depth_to_Potential_Density.ipynb`` does this
    online in MOM; CMIP6 publishes only z-coordinate ``vmo``, so the
    rebinning is offline and uses monthly-mean density.  That
    underestimates the eddy contribution to the transport, which is why
    the diagnostic reports it as a separate method rather than as an
    equivalent of the online diagnostic.
    """
    return bin_sum(transport, density, bins, axis=depth_axis)


def cumulative_from_bottom(values, axis=0):
    """Cumulative sum from the bottom up along ``axis``.

    ``numpy`` sums from index 0 down, so the bottom-up cumulative sum is
    the total minus the top-down sum, exactly as
    ``Overturning_Circulation.ipynb`` writes it.
    """
    values = np.ma.masked_invalid(np.ma.asarray(values, dtype=float))
    filled = np.ma.filled(values, 0.0)
    return np.cumsum(filled, axis=axis) - np.sum(
        filled, axis=axis, keepdims=True)


def check_transport_grid(cube, name="transport"):
    """Warn if an extensive transport looks like it has been regridded.

    ``umo`` and ``vmo`` are *extensive*: kg s-1 through one cell face,
    not a flux per unit area.  Interpolating them and then summing
    scales the total with the number of cells, so a 1x1 to 0.5x0.5
    regrid inflates a transport by roughly four.  Nothing in the
    interpolation complains -- the numbers stay plausible -- so this
    checks the one thing that is visible after the fact: whether the
    cube still sits on the model's own (curvilinear) grid.

    A regular lat/lon grid is not proof of regridding, since a few
    models publish ocean data on one, hence a warning rather than an
    error.
    """
    if cube.coord("latitude").ndim == 1:
        LOGGER.warning(
            "%s is on a regular lat/lon grid. If it was regridded, its "
            "total is wrong: %s is kg s-1 per cell face, and interpolation "
            "does not rescale it for the new cell size. Compute transports "
            "on the native grid, or convert to a flux per unit area before "
            "regridding and multiply by the new face area afterwards.",
            name, name)
        return False
    return True


LATITUDE_COORD_NAMES = ("latitude", "grid_latitude")


def latitude_coord(cube):
    """The cube's latitude coordinate, whatever CMOR called it.

    Raises with the names that *are* present, since a missing latitude
    is nearly always a variable whose CMOR dimension differs rather
    than genuinely unlabelled data.
    """
    for name in LATITUDE_COORD_NAMES:
        if cube.coords(name):
            return cube.coord(name)
    for coord in cube.coords():
        if str(coord.units) in ("degrees_north", "degrees_N", "degree_north"):
            return coord
    raise ValueError(
        f"no latitude-like coordinate on this cube; it has "
        f"{sorted(c.name() for c in cube.coords())}")


def row_latitude(cube):
    """Representative latitude of each grid row.

    The honest label for a quantity summed along the ``i`` index of a
    curvilinear grid, such as an overturning streamfunction.
    :func:`lat_1d` takes one arbitrary column instead, which south of a
    tripolar fold is close to the same thing and north of it is not.

    Reduces to the latitude coordinate itself on a regular grid.
    """
    coord = latitude_coord(cube)
    if coord.ndim == 1:
        return coord.points
    return np.ma.mean(np.ma.masked_invalid(coord.points), axis=1)


@contextlib.contextmanager
def quiet_gsw():
    """Silence gsw's masked-array warning for the enclosed call.

    ``gsw`` emits ``'where' used without 'out', expect uninitialized
    memory in output`` every time it is handed a masked array, which is
    every time here.  The uninitialised values sit only underneath the
    mask and never reach the result -- asserted in
    ``validation/test_cosima_common.py`` -- but the warning fires once
    per call and buries everything else a notebook prints.

    Scoped to this one message so genuine gsw warnings still surface.
    """
    with warnings.catch_warnings():
        warnings.filterwarnings(
            "ignore", message=".*'where' used without 'out'.*",
            category=UserWarning)
        yield


def _require_gsw():
    if gsw is None:
        raise ImportError(
            "gsw (TEOS-10) is required for the density diagnostics; "
            "install it with `pixi add gsw`")


def pressure_from_depth(depth, lat):
    """Sea pressure in dbar from depth in metres (positive down)."""
    _require_gsw()
    depth = np.asarray(depth, dtype=float)
    lat = np.asarray(lat, dtype=float)
    return gsw.p_from_z(-depth, lat)


def absolute_salinity(practical_salinity, pressure, lon, lat):
    """Practical salinity (PSU) to absolute salinity (g/kg)."""
    _require_gsw()
    with quiet_gsw():
        return gsw.SA_from_SP(practical_salinity, pressure, lon, lat)


def conservative_temperature(abs_salinity, potential_temperature):
    """Potential temperature (degC) to conservative temperature (degC)."""
    _require_gsw()
    with quiet_gsw():
        return gsw.CT_from_pt(abs_salinity, potential_temperature)


def _broadcast_to_field(depth, lat, lon, shape):
    """Broadcast depth/lat/lon onto a ``(depth, lat, lon)`` field shape."""
    depth_b = np.broadcast_to(np.asarray(depth)[:, None, None], shape)
    lat_b = np.broadcast_to(np.asarray(lat)[None, :, None], shape)
    lon_b = np.broadcast_to(np.asarray(lon)[None, None, :], shape)
    return depth_b, lat_b, lon_b


def teos10_fields(thetao_cube, so_cube, temperature_is_conservative=False):
    """Absolute salinity, conservative temperature and pressure.

    Accepts fields with optional leading dimensions, such as time:
    ``(lat, lon)``, ``(depth, lat, lon)`` or
    ``(time, depth, lat, lon)``.  It returns masked arrays broadcast to
    the cube shape.  CMIP ``thetao`` is potential temperature, so it is
    converted unless the caller says otherwise (MOM5 ``temp`` is already
    conservative).
    """
    _require_gsw()
    salinity = masked_data(so_cube)
    temperature = masked_data(thetao_cube)

    if salinity.shape != temperature.shape:
        raise ValueError("thetao and so must have the same shape")

    def broadcast_1d(points, axis):
        """Broadcast a coordinate along its corresponding data axis."""
        reshape = [1] * salinity.ndim
        reshape[axis] = len(points)
        return np.broadcast_to(np.asarray(points).reshape(reshape),
                               salinity.shape)

    lat = lat_1d(so_cube)
    lon = lon_1d(so_cube)
    lat_axis = so_cube.coord_dims("latitude")[0]
    lon_axis = so_cube.coord_dims("longitude")[-1]
    depth_name = depth_coord_name(so_cube)
    if depth_name is None:
        depth_b = np.zeros(salinity.shape)
    else:
        depth = so_cube.coord(depth_name).points
        depth_axis = so_cube.coord_dims(depth_name)[0]
        depth_b = broadcast_1d(depth, depth_axis)
    lat_b = broadcast_1d(lat, lat_axis)
    lon_b = broadcast_1d(lon, lon_axis)
    pressure = pressure_from_depth(depth_b, lat_b)
    abs_sal = absolute_salinity(salinity, pressure, lon_b, lat_b)
    if temperature_is_conservative:
        cons_temp = temperature
    else:
        cons_temp = conservative_temperature(abs_sal, temperature)

    return abs_sal, cons_temp, pressure


def potential_density_anomaly(thetao_cube, so_cube, p_ref=2000,
                              temperature_is_conservative=False):
    """Potential density anomaly sigma_n in kg m-3.

    ``p_ref`` is the reference pressure in dbar; 0, 1000, 2000, 3000 and
    4000 map onto the ``gsw.sigma0``...``gsw.sigma4`` family.
    """
    _require_gsw()
    sigma_functions = {
        0: gsw.sigma0,
        1000: gsw.sigma1,
        2000: gsw.sigma2,
        3000: gsw.sigma3,
        4000: gsw.sigma4,
    }
    if p_ref not in sigma_functions:
        raise ValueError(
            f"p_ref must be one of {sorted(sigma_functions)}, got {p_ref}")
    abs_sal, cons_temp, _ = teos10_fields(
        thetao_cube, so_cube,
        temperature_is_conservative=temperature_is_conservative)
    with quiet_gsw():
        return np.ma.masked_invalid(
            sigma_functions[p_ref](abs_sal, cons_temp))

## Density-space streamfunction

Bin `vmo` by the sigma2 of the same cell, sum zonally, convert
mass transport to volume transport with a Boussinesq reference
density, and integrate from the sea floor upwards.

### Two things that differ from the original's figure

**The vertical axis.** This plots $\sigma_2$, the density
*anomaly*; the COSIMA original labels its axis "Potential
Density" and plots $\sigma_2 + 1000$. Same quantity, offset
by 1000 — their 1030 is this notebook's 30. Add 1000 mentally
when comparing.

**How wide the bins are.** MOM6 bins transport online into its
own `rho2` layers, which is why the original's axis runs down to
1010: the coordinate covers brackish water even though almost
none of the ocean sits there. Doing the rebin offline, the same
care is needed for the opposite reason. A cell whose
$\sigma_2$ falls outside the bins contributes to **no** bin,
so its transport vanishes — and because the streamfunction is a
cumulative sum, losing transport offsets that latitude by a
constant at *every* density.

On the section that appears as a **vertical stripe** spanning
the full density range over a band of latitudes. It is easy to
read as a feature; it is missing water. The cell below reports
what fraction of the transport landed inside the bins, and how
close the streamfunction comes to closing at the light end.

In [ ]:
# Bins deliberately far wider than the range worth plotting.
# A cell whose density falls outside them contributes to no
# bin at all, which offsets that latitude by a constant at
# EVERY density -- a vertical stripe across the section.
# Brackish water (river plumes, sea-ice melt, the Baltic)
# reaches sigma2 below 20, so start well below anything the
# open ocean occupies and narrow the view when plotting.
SIGMA2_BINS = np.arange(15.0, 38.51, 0.05)
PLOT_RANGE = (37.8, 29.5)      # (densest, lightest) shown
MAX_LATITUDE = 65.0

sigma2 = potential_density_anomaly(thetao_cube, so_cube,
                                   p_ref=2000)

# Warns if vmo has ended up on a regular grid, which would
# mean its total is no longer the model's transport.
check_transport_grid(vmo_cube, 'vmo')

transport = masked_data(vmo_cube)
binned = rebin_to_density(transport, sigma2, SIGMA2_BINS,
                          depth_axis=0)
zonal = np.ma.sum(np.ma.masked_invalid(binned), axis=-1)
zonal = (zonal / RHO0) / 1.0e6          # kg/s -> Sv

psi_rho = cumulative_from_bottom(zonal, axis=0)
centres = bin_centres(SIGMA2_BINS)
# Row-mean latitude: the honest label for a quantity summed
# along i on a curvilinear grid. Column 0 is one meridian.
lat = row_latitude(vmo_cube)

# Two sanity numbers, both worth reading before the figure.
inside = ((sigma2 >= SIGMA2_BINS[0])
          & (sigma2 <= SIGMA2_BINS[-1]))
weight = np.ma.abs(transport)
kept = (np.ma.sum(np.ma.where(inside, weight, 0.0))
        / np.ma.sum(weight))
residual = np.ma.max(np.ma.abs(psi_rho[0]))

print(f'peak density-space overturning: '
      f'{np.ma.max(np.ma.abs(psi_rho)):.1f} Sv')
print(f'transport inside the bins       : {100*kept:.3f}%  '
      f'(sigma2 spans {np.ma.min(sigma2):.1f} to '
      f'{np.ma.max(sigma2):.1f})')
print(f'closure residual at the light end: {residual:.2f} Sv  '
      f'(a few Sv or more means transport is missing)')

## Depth-space streamfunction

In [ ]:
try:
    msftyz_cube = climate_statistics(msftyz.load(),
                                     operator='mean',
                                     period='full')
    psi_z = masked_data(msftyz_cube)
    while psi_z.ndim > 2:
        psi_z = psi_z[0]          # first basin = global
    if str(msftyz_cube.units) in ('kg s-1', 'kg/s'):
        psi_z = (psi_z / RHO0) / 1.0e6
    depth_z = msftyz_cube.coord(
        depth_coord_name(msftyz_cube)).points
    lat_z = lat_1d(msftyz_cube)
except Exception as error:
    print('msftyz not available:', error)
    psi_z = None

## Plot

In [ ]:
levels = np.arange(-25, 26, 2)
n_panels = 1 if psi_z is None else 2
fig, axes = plt.subplots(n_panels, 1, figsize=(10, 5 * n_panels),
                         squeeze=False)

ax = axes[0, 0]
mesh = ax.contourf(lat, centres, psi_rho, levels=levels,
                   cmap='RdBu_r', extend='both')
ax.contour(lat, centres, psi_rho, levels=levels, colors='k',
           linewidths=0.25)
ax.invert_yaxis()
ax.set_ylim(*PLOT_RANGE)   # bins are wider on purpose
ax.set_xlim(-75, MAX_LATITUDE)
ax.set_ylabel(r'$\sigma_2$ (kg m$^{-3}$)')
ax.set_xlabel('Latitude')
ax.set_title('Density space (offline rebin of monthly vmo:\n'
             'eddy transport underestimated)')
fig.colorbar(mesh, ax=ax, shrink=0.85, label='Overturning (Sv)')

if psi_z is not None:
    ax = axes[1, 0]
    mesh = ax.contourf(lat_z, depth_z, psi_z, levels=levels,
                       cmap='RdBu_r', extend='both')
    ax.contour(lat_z, depth_z, psi_z, levels=levels, colors='k',
               linewidths=0.25)
    ax.invert_yaxis()
    ax.set_xlim(-75, MAX_LATITUDE)
    ax.set_ylabel('Depth (m)')
    ax.set_xlabel('Latitude')
    ax.set_title('Depth space (msftyz)')
    fig.colorbar(mesh, ax=ax, shrink=0.85,
                 label='Overturning (Sv)')

plt.tight_layout()
plt.show()

## Shut the cluster down

Workers hold their memory until the cluster closes. Leaving one running
in an idle kernel is the usual reason a later notebook cannot get the
memory it asks for.

In [ ]:
client.close()